# Fine-tune Legal-BERT on CUAD for Contract Clause Risk Analyzer

This notebook fine-tunes `nlpaueb/legal-bert-base-uncased` into a 42-way clause
classifier (the 41 official CUAD categories + an `Unknown` class) for the
**Contract Clause Risk Analyzer** project.

## Before you start
- In Colab: **Runtime -> Change runtime type -> T4 GPU** (free tier is fine).
- **Runtime -> Run all**. Expected time on a free T4: roughly **1-2 hours**.
- You do **not** need to understand the code. Just run every cell top to bottom.
- If any cell raises an error, **stop and send the full error message (and the
  output of the cell just above it) back to me** — do not try to fix it
  yourself. Everything here is designed to fail loudly with a clear message
  rather than silently produce a bad model.
- Near the end you'll be asked to paste in a free Hugging Face access token.
  Instructions for getting one are in that cell. Never share that token with
  anyone but this notebook.

## What you'll get back
1. A results table (accuracy / precision / recall / F1 per category) — this is
   the "Technical Performance" evaluation the project proposal asks for.
   Copy/screenshot the final table and send it to me.
2. A trained model, either pushed to your own free Hugging Face Hub account
   (recommended — I can then load it directly), or as a downloadable zip file
   (fallback if you don't want to make an HF account).


## 1. Install dependencies

In [ ]:
!pip install -q "transformers>=4.46" "datasets>=3.0" "accelerate>=1.0" scikit-learn huggingface_hub
print("Dependencies installed.")


## 2. Confirm GPU is available
If this prints `No GPU found`, go to **Runtime -> Change runtime type -> T4 GPU** and re-run from the top.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU OK: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found. Go to Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.")


## 3. Download CUAD

Downloads the official CUAD dataset (Hendrycks et al., 2021) directly from the
Atticus Project's GitHub repository as a SQuAD-v2-style JSON file.

In [ ]:
import os, zipfile, urllib.request

DATA_DIR = "/content/cuad_data"
os.makedirs(DATA_DIR, exist_ok=True)
ZIP_PATH = os.path.join(DATA_DIR, "data.zip")
JSON_PATH = None

url = "https://github.com/The-Atticus-Project/cuad/raw/main/data.zip"
print(f"Downloading {url} ...")
urllib.request.urlretrieve(url, ZIP_PATH)
print(f"Downloaded {os.path.getsize(ZIP_PATH) / 1e6:.1f} MB")

with zipfile.ZipFile(ZIP_PATH) as zf:
    names = zf.namelist()
    json_candidates = [n for n in names if n.lower().endswith(".json") and "cuad" in n.lower()]
    if not json_candidates:
        json_candidates = [n for n in names if n.lower().endswith(".json")]
    assert json_candidates, f"No JSON file found inside data.zip. Contents: {names[:20]}"
    JSON_PATH = os.path.join(DATA_DIR, os.path.basename(json_candidates[0]))
    with zf.open(json_candidates[0]) as src, open(JSON_PATH, "wb") as dst:
        dst.write(src.read())

print(f"Extracted: {JSON_PATH}")


## 4. Verify the 41 official CUAD categories

This list was fetched directly from
`github.com/The-Atticus-Project/cuad/blob/main/category_descriptions.csv`
(not from memory) and must match `backend/ml/labels.py` in the main project
exactly, so the checkpoint's label strings line up with the app.

In [ ]:
CUAD_CATEGORIES = [
    "Document Name", "Parties", "Agreement Date", "Effective Date", "Expiration Date",
    "Renewal Term", "Notice Period to Terminate Renewal", "Governing Law",
    "Most Favored Nation", "Non-Compete", "Exclusivity", "No-Solicit of Customers",
    "Competitive Restriction Exception", "No-Solicit of Employees", "Non-Disparagement",
    "Termination for Convenience", "Rofr/Rofo/Rofn", "Change of Control", "Anti-Assignment",
    "Revenue/Profit Sharing", "Price Restrictions", "Minimum Commitment", "Volume Restriction",
    "IP Ownership Assignment", "Joint IP Ownership", "License Grant", "Non-Transferable License",
    "Affiliate License-Licensor", "Affiliate License-Licensee",
    "Unlimited/All-You-Can-Eat-License", "Irrevocable or Perpetual License",
    "Source Code Escrow", "Post-Termination Services", "Audit Rights", "Uncapped Liability",
    "Cap on Liability", "Liquidated Damages", "Warranty Duration", "Insurance",
    "Covenant Not to Sue", "Third Party Beneficiary",
]
assert len(CUAD_CATEGORIES) == 41, f"Expected 41 categories, got {len(CUAD_CATEGORIES)}"

UNKNOWN_LABEL = "Unknown"
ALL_LABELS = CUAD_CATEGORIES + [UNKNOWN_LABEL]  # 42-way classification
LABEL2ID = {name: i for i, name in enumerate(ALL_LABELS)}
ID2LABEL = {i: name for name, i in LABEL2ID.items()}
print(f"{len(ALL_LABELS)} labels total (41 CUAD + Unknown).")


## 5. Parse CUAD and map each question to a category

CUAD's raw format is SQuAD-v2 style: every contract has one `context`
(the full contract text) and a fixed list of 41 `qas` (question/answer)
entries, one per category, always in the same order. Each `qas` entry's `id`
field also encodes the category name after a `__` separator in the official
dataset. To be safe against small format differences, this cell tries **three
independent strategies** and cross-checks them against each other and against
the verified 41-category list above:

1. Category name embedded in the `id` field after `__`.
2. Category name embedded in quotes inside the `question` text.
3. Positional index within each contract's fixed-order `qas` list.

If strategies disagree on more than a handful of examples, the cell prints a
warning with details — if you see that warning, send me the printed output.

In [ ]:
import json, re, collections

with open(JSON_PATH) as f:
    cuad_raw = json.load(f)

data = cuad_raw["data"]
print(f"Loaded {len(data)} contracts.")

def guess_by_id(qa_id: str):
    if "__" in qa_id:
        candidate = qa_id.rsplit("__", 1)[-1].strip()
        return candidate if candidate in CUAD_CATEGORIES else None
    return None

def guess_by_question(question: str):
    m = re.search(r'"([^"]+)"', question)
    if m and m.group(1).strip() in CUAD_CATEGORIES:
        return m.group(1).strip()
    return None

# Positive examples: (text, category). Also collect raw answer char-spans per
# contract so step 6 can build "Unknown" negatives from the untouched text.
positive_examples = []
contract_spans = []  # list of (context, [(start, end), ...])
strategy_votes = collections.Counter()
disagreements = []

for entry in data:
    for para in entry.get("paragraphs", []):
        context = para["context"]
        qas = para["qas"]
        spans_this_contract = []

        positional_ok = len(qas) == 41
        for idx, qa in enumerate(qas):
            id_guess = guess_by_id(qa.get("id", ""))
            q_guess = guess_by_question(qa.get("question", ""))
            pos_guess = CUAD_CATEGORIES[idx] if positional_ok else None

            guesses = [g for g in (id_guess, q_guess, pos_guess) if g]
            if not guesses:
                continue
            category, votes = collections.Counter(guesses).most_common(1)[0]
            strategy_votes[votes] += 1
            if len(set(guesses)) > 1:
                disagreements.append((qa.get("id", ""), id_guess, q_guess, pos_guess))

            for ans in qa.get("answers", []):
                text = ans["text"].strip()
                start = ans["answer_start"]
                if len(text.split()) < 3:
                    continue  # too short to be a useful training example
                positive_examples.append((text, category))
                spans_this_contract.append((start, start + len(text)))

        contract_spans.append((context, spans_this_contract))

print(f"Positive examples collected: {len(positive_examples)}")
print(f"Agreement strength histogram (how many of the 3 strategies agreed): {dict(strategy_votes)}")
if disagreements:
    print(f"\n{len(disagreements)} questions had disagreeing category guesses. First 5:")
    for d in disagreements[:5]:
        print("  ", d)
    if len(disagreements) > 200:
        print("WARNING: large number of disagreements — send this output to me before continuing.")

seen_categories = {c for _, c in positive_examples}
missing = set(CUAD_CATEGORIES) - seen_categories
if missing:
    print(f"\nWARNING: no positive examples found for: {sorted(missing)}")
    print("This is not necessarily fatal (some categories are rare in CUAD) but note it.")


## 6. Build the "Unknown" negative class

Samples paragraph-length windows of contract text that don't overlap any
labeled clause span, so the model learns to distinguish real clause language
from ordinary boilerplate/filler text — mirrors the app's own `Unknown`
fallback category.

In [ ]:
import random
random.seed(42)

WINDOW_WORDS = 220
negative_examples = []

for context, spans in contract_spans:
    covered = bytearray(len(context))
    for start, end in spans:
        for i in range(max(0, start), min(len(context), end)):
            covered[i] = 1

    words = context.split()
    if len(words) < WINDOW_WORDS:
        continue

    # Try a handful of random windows per contract; keep ones mostly uncovered.
    for _ in range(8):
        start_word = random.randint(0, max(0, len(words) - WINDOW_WORDS))
        window_words = words[start_word: start_word + WINDOW_WORDS]
        window_text = " ".join(window_words)
        # Rough overlap check via substring search of the window start.
        char_pos = context.find(window_words[0]) if window_words else -1
        if char_pos == -1:
            continue
        span_covered = covered[char_pos: char_pos + len(window_text)]
        if span_covered and (sum(span_covered) / len(span_covered)) < 0.15:
            negative_examples.append((window_text, "Unknown"))

# Balance roughly against the positive set, capped so training stays fast.
random.shuffle(negative_examples)
target_negatives = min(len(negative_examples), max(1500, len(positive_examples) // 3))
negative_examples = negative_examples[:target_negatives]

print(f"Negative (Unknown) examples: {len(negative_examples)}")

all_examples = positive_examples + negative_examples
random.shuffle(all_examples)
print(f"Total training examples: {len(all_examples)}")

label_counts = collections.Counter(c for _, c in all_examples)
print("\nExamples per category:")
for name in ALL_LABELS:
    print(f"  {name:38s} {label_counts.get(name, 0)}")


## 7. Train/validation split + tokenization

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

texts = [t for t, _ in all_examples]
labels = [LABEL2ID[c] for _, c in all_examples]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.15, random_state=42, stratify=labels
)
print(f"Train: {len(train_texts)}  Val: {len(val_texts)}")

MODEL_ID = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize(batch_texts):
    return tokenizer(batch_texts, truncation=True, max_length=512, padding="max_length")

train_enc = tokenize(train_texts)
val_enc = tokenize(val_texts)

import torch as _torch

class ClauseDataset(_torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: _torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = _torch.tensor(self.labels[idx])
        return item

train_dataset = ClauseDataset(train_enc, train_labels)
val_dataset = ClauseDataset(val_enc, val_labels)


## 8. Fine-tune

Uses class-weighted loss (CUAD categories are naturally imbalanced — e.g.
Governing Law is far more common than Rofr/Rofo/Rofn) and settings sized for
a free-tier T4 GPU (fp16, small batch + gradient accumulation).

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification, Trainer, TrainingArguments,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(ALL_LABELS)),
    y=np.array(train_labels),
)
class_weights_t = _torch.tensor(class_weights, dtype=_torch.float)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=len(ALL_LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = _torch.nn.CrossEntropyLoss(weight=class_weights_t.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {"accuracy": acc, "macro_precision": precision, "macro_recall": recall, "macro_f1": f1}

training_args = TrainingArguments(
    output_dir="/content/legal-bert-cuad-checkpoint",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    fp16=torch.cuda.is_available(),
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to=[],
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
print("Training complete.")


## 9. Evaluate

This table is the technical-performance evaluation the project proposal
asks for. **Copy or screenshot the printed table and send it back to me.**

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

report = classification_report(
    val_labels, preds, target_names=ALL_LABELS, output_dict=True, zero_division=0,
)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv("/content/cuad_classification_report.csv")

print(f"Overall accuracy: {accuracy_score(val_labels, preds):.4f}\n")
print(report_df.round(3).to_string())


## 10. Save the checkpoint locally

In [ ]:
SAVE_DIR = "/content/legal-bert-cuad-final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")


## 11. Hand off the checkpoint (pick ONE option)

### Option A — Push to Hugging Face Hub (recommended)
1. Create a free account at https://huggingface.co/join if you don't have one.
2. Go to https://huggingface.co/settings/tokens and create a new token with
   **write** access.
3. Run the cell below, paste the token when prompted (it will be hidden —
   this is normal), and choose a repo name.
4. When it finishes, it prints a model URL like
   `https://huggingface.co/<your-username>/legal-bert-cuad-clauses` —
   send me that URL.

### Option B — Manual download (if you'd rather not make an HF account)
Skip the cell below and run the one after it instead — it zips the checkpoint
and downloads it through your browser. Send me the zip file (Google Drive
link, WeTransfer, etc. — it will likely be 400-500MB).

In [ ]:
# OPTION A: push to Hugging Face Hub. Skip this cell if you're using Option B.
from huggingface_hub import login, whoami

login()  # paste your HF write-access token when prompted

username = whoami()["name"]
REPO_NAME = f"{username}/legal-bert-cuad-clauses"

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)
print(f"\nDone. Send this URL back: https://huggingface.co/{REPO_NAME}")
print(f"Repo id to use in .env -> FINE_TUNED_MODEL_PATH={REPO_NAME}")


In [ ]:
# OPTION B: zip + browser download. Skip this cell if you used Option A above.
import shutil
from google.colab import files

shutil.make_archive("/content/legal-bert-cuad-final", "zip", SAVE_DIR)
print("Zipped. Starting download (this can take a minute for ~400-500MB)...")
files.download("/content/legal-bert-cuad-final.zip")


## Done — what to send back

1. The evaluation table printed in Step 9 (and/or `cuad_classification_report.csv`,
   downloadable via the Colab file browser on the left).
2. Either the Hugging Face repo URL (Option A) or the downloaded zip file
   (Option B).

Once I have either of those, I'll wire `FINE_TUNED_MODEL_PATH` into the
app's `.env`, restart the backend, and the classifier will use your
fine-tuned checkpoint instead of the keyword/prototype fallback.